# Volume 1. What Is an Assembly?

Question: what changes as sparse activity stabilizes, and what changes again when two traces are bound into a new target assembly?

This first encounter uses a concrete toy scene: a `red` cue forms activity in a COLOR area, a `triangle` cue forms activity in a SHAPE area, and merge binds both source assemblies into an OBJECT area. The plots are index maps and diagnostics, not cortical anatomy.


In [ ]:
import copy

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from neural_assemblies.assembly_calculus import Assembly
from neural_assemblies.assembly_calculus.tracing import (
    merge_trace,
    project_trace,
    source_response_traces,
)
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import (
    animate_assembly_trace,
    plot_assemblies,
    plot_binding_story,
    plot_merge_diagnostic,
    plot_response_overlap,
    plot_trace_metrics,
    plot_winner_turnover,
)


The experiment has one shape: form two source assemblies, watch target activity settle over rounds, and then ask whether either source can partially recover the bound object.


In [ ]:
flow_fig = plot_binding_story(
    stimulus_labels=["red cue\nexternal input", "triangle cue\nexternal input"],
    source_labels=["COLOR area\nwinners", "SHAPE area\nwinners"],
    target_label="OBJECT area\nbound winners",
    title="A toy binding circuit",
    caption="Conceptual routing diagram: useful for orientation, not a brain atlas.",
)
plt.show()
plt.close(flow_fig)


The cast of characters is deliberately small. `N` is the number of neurons in each area, `K` is the sparse winner count, and `BETA` controls how much active co-firing strengthens future projections.


In [ ]:
N = 5_000
K = 80
BETA = 0.08
PROJECTION_ROUNDS = 8
MERGE_ROUNDS = 5
SOURCE_RESPONSE_ROUNDS = 6
SEED = 42

STIMULI = {"red cue": "red_cue", "triangle cue": "triangle_cue"}
AREAS = {
    "COLOR": "source area for the color cue",
    "SHAPE": "source area for the shape cue",
    "OBJECT": "target area for the bound object",
}

brain = Brain(p=0.05, save_winners=True, seed=SEED, engine="numpy_sparse")
for stimulus in STIMULI.values():
    brain.add_stimulus(stimulus, K)
for area in AREAS:
    brain.add_area(area, N, K, BETA)

pd.DataFrame(
    [{"kind": "stimulus", "label": label, "internal_name": name, "k": K}
     for label, name in STIMULI.items()]
    + [{"kind": "area", "label": area, "internal_name": area, "k": K, "note": note}
       for area, note in AREAS.items()]
)


First form the two source traces. The important thing to watch is not just the final dot cloud; it is the trace of overlap and winner turnover as recurrence stabilizes the sparse set.


In [ ]:
red_trace = project_trace(brain, STIMULI["red cue"], "COLOR", rounds=PROJECTION_ROUNDS)
triangle_trace = project_trace(
    brain,
    STIMULI["triangle cue"],
    "SHAPE",
    rounds=PROJECTION_ROUNDS,
)

red_assembly = red_trace.final
triangle_assembly = triangle_trace.final

pd.DataFrame([red_trace.summary(), triangle_trace.summary()])


In [ ]:
fig, animation = animate_assembly_trace(
    red_trace,
    N,
    title="COLOR assembly forming from the red cue",
    color="#e15554",
    interval=650,
)
html = HTML(animation.to_jshtml())
plt.close(fig)
html


In [ ]:
fig, axes = plot_assemblies(
    [red_assembly, triangle_assembly],
    N,
    titles=["COLOR: red cue", "SHAPE: triangle cue"],
    subtitles=[f"{K} winners | density {K / N:.3f}", f"{K} winners | density {K / N:.3f}"],
    colors=["#e15554", "#4d9de0"],
    marker_size=18,
)
plt.show()
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6.8))
plot_trace_metrics(red_trace, axes=axes[0], title="COLOR stabilization", color="#e15554")
plot_trace_metrics(triangle_trace, axes=axes[1], title="SHAPE stabilization", color="#4d9de0")
plt.show()
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
plot_winner_turnover(red_trace, ax=axes[0], title="COLOR winner turnover")
plot_winner_turnover(triangle_trace, ax=axes[1], title="SHAPE winner turnover")
plt.show()
plt.close(fig)


Now merge the two source assemblies into OBJECT. This is still a toy operation: the result is a new sparse winner set driven by two already-formed sources, not a claim that the model understands red triangles.


In [ ]:
object_before = Assembly("OBJECT", [])
object_trace = merge_trace(brain, "COLOR", "SHAPE", "OBJECT", rounds=MERGE_ROUNDS)
object_assembly = object_trace.final

pd.DataFrame([object_trace.summary()])


In [ ]:
fig, animation = animate_assembly_trace(
    object_trace,
    N,
    title="OBJECT assembly forming from COLOR + SHAPE",
    color="#59a14f",
    interval=750,
)
html = HTML(animation.to_jshtml())
plt.close(fig)
html


In [ ]:
sample_size = 12
pd.DataFrame([
    {
        "assembly": "red cue in COLOR",
        "area": red_assembly.area,
        "winners": len(red_assembly),
        "density": len(red_assembly) / N,
        "sample_winners": red_assembly.winners[:sample_size].tolist(),
    },
    {
        "assembly": "triangle cue in SHAPE",
        "area": triangle_assembly.area,
        "winners": len(triangle_assembly),
        "density": len(triangle_assembly) / N,
        "sample_winners": triangle_assembly.winners[:sample_size].tolist(),
    },
    {
        "assembly": "red triangle in OBJECT",
        "area": object_assembly.area,
        "winners": len(object_assembly),
        "density": len(object_assembly) / N,
        "sample_winners": object_assembly.winners[:sample_size].tolist(),
    },
])


The useful inspection question after a merge is: does the target respond differently when driven by one source versus the other, and how far above random same-area overlap are those responses?


In [ ]:
response_diagnostic = source_response_traces(
    brain,
    ["COLOR", "SHAPE"],
    "OBJECT",
    reference=object_assembly,
    labels=["COLOR alone", "SHAPE alone"],
    rounds=SOURCE_RESPONSE_ROUNDS,
)

pd.DataFrame(response_diagnostic.to_records())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
plot_response_overlap(
    object_assembly,
    [response.final for response in response_diagnostic.responses],
    n=N,
    labels=[response.label for response in response_diagnostic.responses],
    ax=axes[0],
    title="Single-source OBJECT responses",
)
plot_winner_turnover(object_trace, ax=axes[1], title="OBJECT winner turnover")
plt.show()
plt.close(fig)


In [ ]:
replay_brain = copy.deepcopy(brain)
replay_trace = merge_trace(replay_brain, "COLOR", "SHAPE", "OBJECT", rounds=MERGE_ROUNDS)

diagnostic_fig = plot_merge_diagnostic(
    object_before,
    object_assembly,
    replay_trace.final,
    n=N,
    title="Before, after, and same-pair replay",
)
plt.show()
plt.close(diagnostic_fig)


What this notebook establishes: assemblies are sparse sets, projection can stabilize a trace, merge creates a target set driven by two source traces, and every visual claim here is backed by overlap or winner diagnostics. What it does not establish: semantics, perception, or image understanding.
